# Module 1: Data Input and Gaia Survival Test

This module accepts and validates input data (vetted K Dwarfs or virgin coordinates) and performs the Gaia DR3 Survival Test.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import sys
import os
import tempfile
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Add src to path
sys.path.append(os.path.join(os.path.dirname(''), '..', 'src'))

st.set_page_config(
    page_title="Module 1: Data Input and Gaia Survival Test",
    page_icon="📥",
    layout="wide",
    initial_sidebar_state="collapsed",
)

In [ ]:
# Load CSS
def local_css(file_name):
    with open(file_name) as f:
        st.markdown(f'<style>{f.read()}</style>', unsafe_allow_html=True)

local_css("streamlit_app/static/style.css")

In [ ]:
# Page header
st.markdown("""
<div style="text-align: center; margin-top: 0.5rem; margin-bottom: 1rem;">
    <div style="font-size: 3rem; line-height: 1;">📥</div>
    <h1 style="margin: 0.25rem 0 0 0; font-weight: 700; font-size: 1.5rem;">
        Module 1: Data Input and Gaia Survival Test
    </h1>
</div>
""", unsafe_allow_html=True)

st.markdown("---")

# Import module logic
from modules.module1_data_input import DataInputModule
from modules.integrity_tracker import IntegrityTracker

from workspace import current_user, get_store
from workspace.store import normalize_user_id, RunMeta, RunRecord, new_run_id
from identifier_resolver import parse_manual_input
from gaia_enricher import enrich_rows

st.markdown("""
**The starting line of your quest for Earth 2.0.**  
Hand the pipeline a list of sky coordinates — upload a CSV or type 
RA/Dec pairs by hand.
""")

with st.expander("READ MORE: The Input Process . . ."):
    st.markdown("""
    Module 1 accepts any CSV with `ra` and `dec` columns. Validated K Dwarf 
    catalogs (those with `DR3Name`, `Teff`, `logg`, `RUWE`, etc.) are 
    auto-recognized and routed through the rich loader. Manual Entry lets 
    you type RA/Dec pairs by hand for quick spot checks.

    After ingest, Module 1 sanity-checks coordinate ranges, deduplicates 
    by source ID where available, standardizes column names, and prepares 
    the batch for the **Gaia DR3 Survival Test** below. 
    Only the stars that survive move on to Modules 2–8.
    """)

st.caption(
    "Type RA/Dec pairs or catalog IDs below — or use the **📂 Upload CSV** widget on the right. "
    "Accepts: **RA / Dec** *(required)*, **Gaia DR3 IDs**, **TIC IDs**, "
    "**KIC / EPIC IDs**, **2MASS / Spitzer IDs**. Extra columns "
    "(Teff, logg, RUWE, photometry, …) are auto-recognized."
)

In [ ]:
# Data source selection
st.markdown("### Data Source")

data_source = st.radio(
    "Choose input method:",
    ["Upload CSV", "Manual Entry"],
    horizontal=True,
    label_visibility="collapsed"
)

In [ ]:
# CSV upload
if data_source == "Upload CSV":
    uploaded_file = st.file_uploader(
        "Upload CSV file with coordinates",
        type=['csv'],
        help="CSV with RA/Dec columns or Gaia DR3 catalog"
    )
    
    n_stars = st.slider(
        "Number of stars to process",
        min_value=1,
        max_value=1000,
        value=10,
        step=1
    )
    
    random_sample = st.checkbox("Random sample", value=False)
    
    if uploaded_file is not None:
        preview_df = pd.read_csv(uploaded_file)
        st.subheader("Preview")
        st.dataframe(preview_df.head(10))

In [ ]:
# Manual entry
if data_source == "Manual Entry":
    st.markdown("""
    **Enter coordinates or identifiers** (one per line):
    - RA, Dec in decimal degrees
    - Gaia DR3 source ID
    - TIC ID (TESS Input Catalog)
    - KIC ID (Kepler)
    - EPIC ID (K2)
    - 2MASS, HD, HIP, TYC catalog IDs
    """)
    
    manual_text = st.text_area(
        "Coordinates or identifiers",
        height=200,
        placeholder="Example:\n10.5, 20.3\nGaia DR3 1234567890123456784\nTIC 123456789"
    )

In [ ]:
# Run Module 1 button
st.markdown("---")

col1, col2 = st.columns([1, 3])
with col1:
    run_module = st.button("▶️ Run Module 1", type="primary")
with col2:
    st.caption("Load and validate input data through Gaia DR3 Survival Test")

In [ ]:
# Module execution
if run_module:
    module1 = DataInputModule()
    tracker = IntegrityTracker()
    
    if data_source == "Upload CSV":
        if uploaded_file is None:
            st.error("Please upload a CSV file before running Module 1.")
            st.stop()
        
        # Persist upload to temp file
        with tempfile.NamedTemporaryFile(delete=False, suffix=".csv") as tmp:
            tmp.write(uploaded_file.getvalue())
            tmp_path = tmp.name
        
        # Load CSV
        head = pd.read_csv(tmp_path, nrows=1)
        if "DR3Name" in head.columns or "validation_tier" in head.columns:
            df, validation = module1.load_real_kdwarf_catalog(
                file_path=tmp_path,
                n_stars=n_stars,
                random_sample=random_sample,
            )
        else:
            df, validation = module1.load_csv(tmp_path)
            if n_stars and len(df) > n_stars:
                df = df.head(n_stars).reset_index(drop=True)
                module1.data = df
    else:
        # Manual entry
        with st.spinner("Resolving identifiers via Simbad / MAST …"):
            rows, unresolved = parse_manual_input(manual_text)
        
        if unresolved:
            st.warning(
                f"Could not resolve {len(unresolved)} line(s): "
                + ", ".join(f"`{u}"" for u in unresolved[:6])
                + (" …" if len(unresolved) > 6 else "")
            )
        
        if not rows:
            st.error(
                "No valid coordinates or identifiers found. "
                "Each line must be 'RA, Dec' in decimal degrees "
                "or a catalog ID (Gaia DR3 / TIC / KIC / EPIC / 2MASS / HD / HIP / TYC)."
            )
            st.stop()
        
        coordinates = [{"ra": r["ra"], "dec": r["dec"]} for r in rows]
        df, validation = module1.load_manual_entry(coordinates)
        
        # Gaia DR3 enrichment
        progress_bar = st.progress(0.0, text="Cross-matching against live Gaia DR3 …")
        
        def _on_gaia_progress(done, total, msg):
            pct = min(done / max(total, 1), 1.0)
            progress_bar.progress(pct, text=msg)
        
        enriched = enrich_rows(rows, progress_cb=_on_gaia_progress)
        progress_bar.empty()
        
        if len(enriched) == len(df):
            enriched = enriched.reset_index(drop=True)
            df = df.reset_index(drop=True)
            for col in [
                "ra", "dec", "parallax", "ruwe",
                "phot_g_mean_mag", "bp_rp",
                "teff_gspphot", "logg_gspphot",
                "gaia_match_arcsec", "gaia_dr3_name", "identifier",
            ]:
                if col in enriched.columns:
                    df[col] = enriched[col]
            if "source_id" in enriched.columns:
                real_sid = enriched["source_id"]
                if "source_id" in df.columns:
                    df["source_id"] = real_sid.where(real_sid.notna(), df["source_id"])
                else:
                    df["source_id"] = real_sid
            module1.data = df
    
    # Initialize integrity tracking
    df = tracker.initialize_integrity_columns(df)
    
    # Mark as Module 1 complete
    df = tracker.mark_module_complete(df, 1)
    
    # Display results
    summary = module1.get_success_summary()
    st.success(summary)
    
    st.subheader("Results")
    st.dataframe(df.head(20), use_container_width=True)

In [ ]:
# Save to workspace
if run_module:
    st.markdown("---")
    st.markdown("### Save to Workspace")
    
    user = current_user()
    if user:
        dataset_name = st.text_input(
            "Dataset name",
            value=f"module1_results_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}"
        )
        
        if st.button("💾 Save to Workspace"):
            try:
                # Save to workspace
                store = get_store()
                uid = normalize_user_id(user)
                
                # Create workspace directory if needed
                workspace_dir = os.path.join("data", "users", uid)
                os.makedirs(workspace_dir, exist_ok=True)
                
                # Save CSV
                file_path = os.path.join(workspace_dir, f"{dataset_name}.csv")
                df.to_csv(file_path, index=False)
                
                st.success(f"Saved {len(df)} coordinates to workspace as '{dataset_name}.csv'")
            except Exception as exc:
                st.error(f"Failed to save: {exc}")
    else:
        st.caption("Sign in to save your results to workspace.")

In [ ]:
# Navigation
st.markdown("---")
st.markdown("### Next Steps")

col1, col2 = st.columns(2)
with col1:
    if st.button("🪐 Go to Module 2"):
        st.info("Opening Module 2: Start Exoplanet Quest...")
        # TODO: Link to module2_exoplanet_crossmatch.ipynb
with col2:
    if st.button("🏠 Return to Navigation Hub"):
        st.info("Returning to Navigation Hub...")
        # TODO: Link to Home.ipynb